In [1]:
import os, duckdb, pandas as pd
os.chdir(os.path.expanduser("~/growth-marketing-analytics"))
con = duckdb.connect("data/processed/criteo.duckdb")
print("connected")

connected


In [2]:
con.execute("""
CREATE OR REPLACE TABLE raw AS
SELECT * FROM read_csv_auto('data/raw/criteo/criteo_attribution_dataset.tsv.gz',
                            delim='\t', header=true)
""")
con.sql("SELECT COUNT(*) AS rows FROM raw").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┐
│   rows   │
│  int64   │
├──────────┤
│ 16468027 │
└──────────┘



In [3]:
con.sql("DESCRIBE raw").df()

,column_name,column_type,null,key,default,extra
0,timestamp,BIGINT,YES,None,None,None
1,uid,BIGINT,YES,None,None,None
2,campaign,BIGINT,YES,None,None,None
3,conversion,BIGINT,YES,None,None,None
4,conversion_timestamp,BIGINT,YES,None,None,None
5,conversion_id,BIGINT,YES,None,None,None
6,attribution,BIGINT,YES,None,None,None
7,click,BIGINT,YES,None,None,None
8,click_pos,BIGINT,YES,None,None,None
9,click_nb,BIGINT,YES,None,None,None


In [4]:
con.sql("""
SELECT
  COUNT(*)                                                     AS impressions,
  COUNT(DISTINCT uid)                                          AS users,
  COUNT(DISTINCT campaign)                                     AS campaigns,
  SUM(click)                                                   AS clicks,
  ROUND(SUM(click)*100.0/COUNT(*), 2)                          AS ctr_pct,
  COUNT(DISTINCT CASE WHEN conversion_id <> -1 THEN conversion_id END) AS conversions,
  ROUND(MAX(timestamp)/86400.0, 1)                             AS days_covered
FROM raw
""").df()

,impressions,users,campaigns,clicks,ctr_pct,conversions,days_covered
0,16468027,6142256,675,5947563.0,36.12,435810,30.9


In [5]:
profile = con.sql("SUMMARIZE raw").df()
profile.to_csv("module_1_criteo/outputs/data_profile.csv", index=False)
profile[["column_name","min","max","avg","null_percentage"]]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,min,max,avg,null_percentage
0,timestamp,0,2671199,1315439.7525673234,0.0
1,uid,13,32458754,16237589.216772234,0.0
2,campaign,73322,32452111,16983964.58268219,0.0
3,conversion,0,1,0.04895522699835263,0.0
4,conversion_timestamp,-1,5262888,94940.22091043451,0.0
5,conversion_id,-1,32458519,793885.2297718482,0.0
6,attribution,0,1,0.02686563484502424,0.0
7,click,0,1,0.3611582006757701,0.0
8,click_pos,-1,173,-0.8312658826707049,0.0
9,click_nb,-1,174,-0.6626016583528798,0.0


In [6]:
con.sql("""
SELECT
  SUM(CASE WHEN conversion=1 AND conversion_id=-1 THEN 1 ELSE 0 END) AS conv_flag_but_no_id,
  SUM(CASE WHEN conversion=0 AND conversion_id<>-1 THEN 1 ELSE 0 END) AS id_but_no_conv_flag,
  SUM(CASE WHEN cost<=0 THEN 1 ELSE 0 END)                            AS zero_or_neg_cost,
  SUM(CASE WHEN conversion_timestamp <> -1
            AND conversion_timestamp < timestamp THEN 1 ELSE 0 END)   AS conv_before_impression
FROM raw
""").df()

,conv_flag_but_no_id,id_but_no_conv_flag,zero_or_neg_cost,conv_before_impression
0,0.0,0.0,0.0,0.0


In [7]:
con.sql("""
SELECT campaigns_in_journey, COUNT(*) AS conversions
FROM (SELECT conversion_id, COUNT(DISTINCT campaign) AS campaigns_in_journey
      FROM raw WHERE conversion_id <> -1
      GROUP BY conversion_id)
GROUP BY 1 ORDER BY 1
""").df()

,campaigns_in_journey,conversions
0,1,432951
1,2,2849
2,3,10


In [8]:
con.sql("""
SELECT campaigns_seen, COUNT(*) AS converting_users
FROM (SELECT uid, COUNT(DISTINCT campaign) AS campaigns_seen
      FROM raw
      WHERE uid IN (SELECT DISTINCT uid FROM raw WHERE conversion_id <> -1)
      GROUP BY uid)
GROUP BY 1 ORDER BY 1 LIMIT 15
""").df()

,campaigns_seen,converting_users
0,1,197572
1,2,85583
2,3,29770
3,4,10018
4,5,3253
5,6,1115
6,7,350
7,8,124
8,9,46
9,10,15
